<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 1. Aspectos generales — Modelo 3: Churn Analysis (Análisis de Abandono)

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Responsable: **Angelina**.

Implementa un modelo **predictivo de abandono (Churn)** en **Spark MLlib** evitando *data leakage* mediante corte temporal:

- **Ventana de observación (features):** `Fecha <= 2020-06-22`.
- **Ventana de etiqueta (churn = 1 si NO compra):** `2020-06-23` a `2020-09-22` (últimos 90 días del dataset).
- **Pipeline MLlib:** `VectorAssembler` -> `StandardScaler` -> `LogisticRegression` & `RandomForestClassifier`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import matplotlib.pyplot as plt

In [ ]:
spark = (
    SparkSession.builder
    .appName("HYM_Lab13_03_Churn")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.memory.fraction", "0.8")
    .getOrCreate()
)

spark.sparkContext.setCheckpointDir("/tmp/spark_checkpoint")
spark

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 2. Rutas de capas

In [ ]:
SILVER = "../silver"
GOLD   = "../gold"

import os
os.makedirs(GOLD, exist_ok=True)

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 3. Construcción de cohorte de entrenamiento sin data leakage

In [ ]:
%%time
df_silver = spark.read.parquet(f"{SILVER}/merge_pyspark").select(
    "customer_id", "Fecha", "price", "sales_channel_id", "age"
)

CORTE = "2020-06-22"
df_obs = df_silver.filter(F.col("Fecha") <= F.lit(CORTE))
df_fut = df_silver.filter(F.col("Fecha") > F.lit(CORTE)).select("customer_id").distinct().withColumn("compro_futuro", F.lit(1))

df_features = (
    df_obs.groupBy("customer_id")
    .agg(
        F.datediff(F.lit(CORTE), F.max("Fecha")).cast("double").alias("recency_obs"),
        F.datediff(F.max("Fecha"), F.min("Fecha")).cast("double").alias("length_obs"),
        F.count("*").cast("double").alias("freq_obs"),
        F.sum("price").cast("double").alias("monetary_obs"),
        F.avg("price").cast("double").alias("aov_obs"),
        F.countDistinct("Fecha").cast("double").alias("dias_activos"),
        F.avg(F.when(F.col("sales_channel_id") == 2, 1.0).otherwise(0.0)).alias("ratio_online"),
        F.coalesce(F.max("age"), F.lit(30)).cast("double").alias("age_clean"),
    )
)

df_dataset = (
    df_features
    .join(df_fut, on="customer_id", how="left")
    .withColumn("label", F.when(F.col("compro_futuro").isNull(), F.lit(1.0)).otherwise(F.lit(0.0)))
    .drop("compro_futuro")
).checkpoint()

print(f"Clientes en ventana de observación: {df_dataset.count():,}")
df_dataset.groupBy("label").count().show()

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 4. Entrenamiento y comparación en MLlib (Train / Test Split 80/20)

In [ ]:
%%time
feature_cols = [
    "recency_obs", "length_obs", "freq_obs", "monetary_obs",
    "aov_obs", "dias_activos", "ratio_online", "age_clean",
]
train_df, test_df = df_dataset.randomSplit([0.8, 0.2], seed=42)

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False)

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=30, maxDepth=6, seed=42)

eval_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
eval_f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")

modelos = {}
for nombre, clf in [("LogisticRegression", lr), ("RandomForest", rf)]:
    pipe = Pipeline(stages=[assembler, scaler, clf])
    model = pipe.fit(train_df)
    preds = model.transform(test_df)
    auc = eval_auc.evaluate(preds)
    f1 = eval_f1.evaluate(preds)
    modelos[nombre] = (model, preds, auc, f1)
    print(f"{nombre:<20} | AUC-ROC = {auc:.4f} | F1 = {f1:.4f}")

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 5. Tasa de Churn por banda de Recency y guardado en Gold

In [ ]:
best_name = max(modelos, key=lambda k: modelos[k][2])
best_model, _, best_auc, _ = modelos[best_name]
print(f"Mejor modelo seleccionado: {best_name} (AUC={best_auc:.4f})")

df_pred_full = best_model.transform(df_dataset).select(
    "customer_id", "recency_obs", "freq_obs", "monetary_obs", "label", "prediction"
)

df_bandas = (
    df_dataset
    .withColumn(
        "banda_recency",
        F.when(F.col("recency_obs") <= 30, "0-30d")
         .when(F.col("recency_obs") <= 90, "31-90d")
         .when(F.col("recency_obs") <= 180, "91-180d")
         .otherwise("180+d"),
    )
    .groupBy("banda_recency")
    .agg(
        F.count("*").alias("clientes"),
        F.round(F.avg("label") * 100.0, 1).alias("tasa_churn_pct"),
    )
    .orderBy("tasa_churn_pct")
)
df_bandas.show()

pdf_b = df_bandas.toPandas()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(pdf_b["banda_recency"], pdf_b["tasa_churn_pct"], color="#B91C1C")
ax.set_title("Tasa Real de Abandono (90 días) según Recency en Ventana de Observación")
ax.set_ylabel("% Churn")
plt.tight_layout()
plt.show()

df_pred_full.write.mode("overwrite").parquet(f"{GOLD}/hym_churn_predicciones")
print(f"Predicciones guardadas en {GOLD}/hym_churn_predicciones")

In [ ]:
spark.stop()